# Fraud Detection Analytics

## Project Objective

The objective of this project is to analyze financial transaction data, identify fraud patterns, and build a machine learning model that can help detect suspicious transactions.

## Business Problem

Fraudulent transactions can cause financial loss for banks, fintech companies, and customers. This project explores transaction behavior to understand which factors are linked with fraud.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [3]:
# Load Dataset
df = pd.read_csv("fraud.csv")

# Display first 5 rows
df.head()

,transaction_amount,hour_of_day,is_weekend,num_items,customer_age,prev_transactions,distance_from_home,device_type,network_quality,is_first_transaction,store_type,velocity_score,is_fraud
0,161.363691,3.0,0.0,2.0,18.000000,2.0,26.539742,1.0,48.403937,0.0,0.0,3.718296,0
1,116.202851,1.0,1.0,4.0,26.285818,2.0,50.714402,NaN,76.144979,0.0,0.0,4.951272,0
2,1.000000,2.0,0.0,5.0,18.000000,NaN,9.467935,0.0,67.600316,0.0,0.0,4.556043,0
3,48.780618,2.0,0.0,3.0,44.471190,NaN,41.077068,0.0,94.825526,0.0,0.0,6.918437,0
4,NaN,3.0,0.0,4.0,38.733609,8.0,NaN,2.0,100.000000,0.0,1.0,5.535335,1


# Data Understanding

## Dataset Structure

Before cleaning or analyzing the data, it is important to understand the dataset structure.

This step helps us:

- Determine the number of records available
- Identify the number of features
- Review all available columns
- Understand which variables may be useful for fraud analysis

In [7]:
print(df.shape)
print(df.columns.tolist())

(7000, 13)
['transaction_amount', 'hour_of_day', 'is_weekend', 'num_items', 'customer_age', 'prev_transactions', 'distance_from_home', 'device_type', 'network_quality', 'is_first_transaction', 'store_type', 'velocity_score', 'is_fraud']
ERROR! Session/line number was not unique in database. History logging moved to new session 6


# Fraud Distribution Analysis

Before building any machine learning model, it is important to understand the distribution of the target variable.

In [9]:
df['is_fraud'].value_counts()

is_fraud
0    6279
1     721
Name: count, dtype: int64

## Column Description

The following features are available in the dataset and may help identify patterns associated with fraudulent transactions.

| Column | Description |
|----------|-------------|
| transaction_amount | Amount spent during the transaction |
| hour_of_day | Hour when the transaction occurred (0–23) |
| is_weekend | Indicates whether the transaction occurred on a weekend |
| num_items | Number of items purchased |
| customer_age | Age of the customer |
| prev_transactions | Number of previous transactions made by the customer |
| distance_from_home | Distance between customer location and transaction location |
| device_type | Device used for the transaction |
| network_quality | Quality of the network connection during the transaction |
| is_first_transaction | Indicates whether this is the customer's first transaction |
| store_type | Type of store where the transaction occurred |
| velocity_score | Measures how quickly transactions are occurring |
| is_fraud | Target variable (0 = Legitimate, 1 = Fraudulent) |



### Initial Observations

- The dataset contains 7,000 transaction records and 13 columns.
- The target variable is **is_fraud**.
- The dataset contains both customer-related and transaction-related features.
- Features such as transaction amount, distance from home, and velocity score may have strong relationships with fraudulent activity.
- Further analysis is required to identify which variables contribute most to fraud detection.

# Data Quality Assessment

Before analyzing the data, it is important to assess its quality.

This step helps identify:

- Missing values
- Incorrect data types
- Duplicate records
- Potential data quality issues

Ensuring data quality improves the accuracy and reliability of any analysis and machine learning model.


## Missing Values Analysis

Missing values can impact both analysis and model performance.

This step helps determine whether any columns contain null values and whether data cleaning is required.

In [11]:
# Check Missing Values

missing_values = df.isnull().sum()

print("Missing Values by Column:")
print(missing_values)

Missing Values by Column:
transaction_amount       560
hour_of_day              350
is_weekend               140
num_items                210
customer_age             840
prev_transactions        490
distance_from_home       700
device_type              280
network_quality          630
is_first_transaction     210
store_type               140
velocity_score          1050
is_fraud                   0
dtype: int64


### Observation

- Multiple columns contain missing values.
- The target variable `is_fraud` contains no missing values.
- `velocity_score` has the highest number of missing values.
- Missing values must be handled before building any machine learning model.

In [12]:
df.dtypes

transaction_amount      float64
hour_of_day             float64
is_weekend              float64
num_items               float64
customer_age            float64
prev_transactions       float64
distance_from_home      float64
device_type             float64
network_quality         float64
is_first_transaction    float64
store_type              float64
velocity_score          float64
is_fraud                  int64
dtype: object

### Observation

- Most columns are stored as numeric data types.
- The target variable `is_fraud` is correctly stored as an integer.
- No obvious data type issues were identified.
- Missing values likely caused some columns to be stored as float data types.

## Duplicate Records Check

Identify whether duplicate transaction records exist in the dataset.

In [14]:
# Check Duplicate Records

duplicates = df.duplicated().sum()

print("Number of Duplicate Records:", duplicates)

Number of Duplicate Records: 0


### Observation

- No duplicate records were found in the dataset.
- Each transaction appears to be unique.
- No duplicate removal is required before analysis.

# Data Cleaning

## Missing Value Treatment

Missing values can affect analysis and machine learning performance.

This step replaces missing values with appropriate measures to retain as much information as possible.

In [15]:
# Percentage of Missing Values

missing_percent = (df.isnull().sum() / len(df)) * 100

missing_percent.sort_values(ascending=False)

velocity_score          15.0
customer_age            12.0
distance_from_home      10.0
network_quality          9.0
transaction_amount       8.0
prev_transactions        7.0
hour_of_day              5.0
device_type              4.0
num_items                3.0
is_first_transaction     3.0
is_weekend               2.0
store_type               2.0
is_fraud                 0.0
dtype: float64

### Observation

- Missing values range from 0% to 15%.
- `velocity_score` has the highest missing percentage (15%).
- No column has excessive missing data.
- Missing values will be imputed rather than removing records.

## Statistical Summary

Review the distribution of numerical features before selecting an imputation strategy.

In [17]:
df.describe()

,transaction_amount,hour_of_day,is_weekend,num_items,customer_age,prev_transactions,distance_from_home,device_type,network_quality,is_first_transaction,store_type,velocity_score,is_fraud
count,6440.000000,6650.000000,6860.000000,6790.000000,6160.000000,6510.000000,6300.000000,6720.000000,6370.000000,6790.000000,6860.000000,5950.000000,7000.000000
mean,100.014648,2.200902,0.108601,2.977320,36.200811,4.457604,24.830735,0.698810,74.094739,0.095876,0.286735,5.007450,0.103000
std,49.091009,0.677906,0.311160,1.712444,13.283996,4.881635,24.314751,0.776802,18.003855,0.294443,0.452270,2.007633,0.303981
min,1.000000,1.000000,0.000000,0.000000,18.000000,0.000000,0.004282,0.000000,0.477174,0.000000,0.000000,-1.853966,0.000000
25%,65.711569,2.000000,0.000000,2.000000,25.244168,1.000000,7.387315,0.000000,61.586731,0.000000,0.000000,3.697990,0.000000
50%,99.482201,2.000000,0.000000,3.000000,35.361187,3.000000,17.321496,1.000000,75.306346,0.000000,0.000000,4.992099,0.000000
75%,132.897923,3.000000,0.000000,4.000000,45.500089,6.000000,34.857859,1.000000,88.253574,0.000000,1.000000,6.342852,0.000000
max,296.311885,3.000000,1.000000,13.000000,80.000000,40.000000,224.699757,2.000000,100.000000,1.000000,1.000000,12.659564,1.000000


### Observation

- Most numerical features have reasonable distributions.
- `transaction_amount` is centered around 100.
- `customer_age` ranges from 18 to 80 years.
- `distance_from_home` shows a wide range of values and may contain outliers.
- Median values appear more reliable than mean values for several features.

## Missing Value Imputation

Missing values are replaced using the median value of each numerical feature to reduce the impact of outliers.

In [19]:
# Fill Missing Values using Median

df_cleaned = df.copy()

for column in df_cleaned.columns:
    if df_cleaned[column].isnull().sum() > 0:
        df_cleaned[column] = df_cleaned[column].fillna(
            df_cleaned[column].median()
        )

print("Missing Values After Cleaning:")
print(df_cleaned.isnull().sum())

Missing Values After Cleaning:
transaction_amount      0
hour_of_day             0
is_weekend              0
num_items               0
customer_age            0
prev_transactions       0
distance_from_home      0
device_type             0
network_quality         0
is_first_transaction    0
store_type              0
velocity_score          0
is_fraud                0
dtype: int64


### Observation

- All missing values were successfully imputed using median values.
- No records were removed from the dataset.
- The cleaned dataset now contains complete information for all variables.

In [23]:
df_cleaned['is_fraud'].value_counts(normalize=True) * 100

is_fraud
0    89.7
1    10.3
Name: proportion, dtype: float64

In [20]:
df_cleaned.to_csv("fraud_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### Observation

- The dataset contains 6,279 legitimate transactions and 721 fraudulent transactions.
- Fraudulent transactions represent only 10.3% of the dataset.
- Legitimate transactions account for 89.7% of the dataset.
- This indicates a class imbalance problem, which is common in fraud detection datasets.